<a href="https://colab.research.google.com/github/NehaReddy79/local-chatbot/blob/main/Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q "transformers<5" accelerate gradio pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [8]:
from pypdf import PdfReader
reader = PdfReader("/content/sample-pdf-book.pdf")
pages = reader.pages
texts = ""
texts += "\n".join(page.extract_text() for page in pages)
print("no page " , len(pages))
print("char len " , len(texts))
print("first 500 char : " , texts[2000:3000])

no page  21
char len  35635
first 500 char :  ightly into life.

Todd sets the bottles by the kitchen sink, washes his hands and looks at
the window in front of him. They could almost be looking at each other, eye
to eye, but Todd is looking at his reflection. Lifting his head, he turns it
from side to side and presses the soft flesh under his chin. Frowning, he
dries his hands and crosses the kitchen to fill the dishwasher. He looks back
towards the hallway. ‘Lucas!’ he calls once more. He waits, but the boy
does not respond.
Todd leaves the dishwasher, walks down the hallway and enters Lucas’s
room without knocking. He taps him on the shoulder. The boy jolts in
fright.
He cannot hear the conversation but gets the gist from Todd’s angry
face. The man points his fingers at the video screen and at the boy’s
homework. Lucas shrugs, switches off the monitor and gets up from his
chair. He leaves the room without looking at his father.
The bathroom light switches on. Through the frosted gla

In [14]:
def chunk_text(text , chunk_size , overlap):
  chunks = []
  for start in range(0 , len(text) , chunk_size - overlap):
    chunks.append(text[start : start+chunk_size])
  return chunks

In [13]:
chunks = chunk_text(texts , 500 , 100)
print(len(chunks))
print(chunks[-1])
print("CHUNK 1")
print(chunks[-2])

90
up on this murder in south
London.’
CHUNK 1
. Let’s take things one step at a time. Please feel free to call
the neurology ward, if you have any concerns.’
‘Thank you, Doctor Prasad.’
‘You’re welcome.’ Prasad hurries off.
‘He sounds positive,’ says Quinn.
Archer is not so sure. ‘Thanks for being here, Harry.’
‘No worries.’
Archer finishes the last of the water and drops the bottle in the waste
bin. ‘We should get back to work. Let’s follow up on this murder in south
London.’


In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks , show_progress_bar=True)



Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [17]:
embeddings.shape

(90, 384)

In [19]:

ques_embedding = model.encode("What is jason todd watching on TV?")
print(ques_embedding.shape)


(384,)


In [29]:
from sentence_transformers import util
import torch
score = util.cos_sim(ques_embedding , embeddings)
print(score.shape)

torch.Size([1, 90])


In [32]:
import torch
top = torch.topk(score[0], k=3)
for score, index in zip(top.values, top.indices):
    print(f"Score: {score:.3f} | Chunk #{index}")
    print("Chunk:", chunks[index])
    print("-"*40)

Score: 0.584 | Chunk #1
Chunk: ground-floor flat. Double doors lead into a dimly lit
modern white kitchen. A stack of dirty dishes from the evening’s meal lies
on the worktop above a partially opened dishwasher. A flat archway leads
through to the living room where the green of an expansive television
broadcasting Match of the Day silhouettes Jason Todd’s head. He’s half
watching the football and half focused on the screen of his mobile phone.
The evening is close and muggy. The forecast predicts a downpour. He
must be quick 
----------------------------------------
Score: 0.559 | Chunk #3
Chunk: all head, cover his ears. He
is pointing a pistol at a computer monitor depicting an unending influx of
bloodthirsty undead hurling themselves at the screen. Lucas has a talent for
conquering them.
He turns his attention back to Jason Todd. He is talking on his mobile.
Todd’s arm reaches up and across and strokes his bald head. His shoulders
shake. He is laughing. The call ends, Todd switches 

In [33]:
from transformers import pipeline
text_gen = pipeline("text-generation",model="Qwen/Qwen2.5-1.5B-Instruct" , device_map = "auto" , torch_dtype = "auto")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [34]:
res = text_gen([{"role" : "user" , "content" : "Hi"}] , max_new_tokens=20)

In [35]:
res

[{'generated_text': [{'role': 'user', 'content': 'Hi'},
   {'role': 'assistant', 'content': 'Hello! How can I assist you today?'}]}]

In [45]:
from sentence_transformers import util
import torch
def retrieve(question , k = 3):
  ques_embedding = model.encode(question)

  #print(ques_embedding.shape)
  score = util.cos_sim(ques_embedding , embeddings)

  #print(score.shape)
  top = torch.topk(score[0], k=k)

  return [chunks[i.item()] for i in top.indices]

In [46]:
def answer(question):
  retrieved = retrieve(question)
  context = "\n---\n".join(retrieved)
  message = [
      {"role" : "system" , "content" : "You are an assistant who will only answer using the provided context, and say 'I don't know based on the document' if the answer isn't there"},
      {"role" : "user" , "content" : f"Context:\n{context}\n\nQuestion: {question}"}

  ]
  result = text_gen(message , max_new_tokens = 200 , do_sample = True , temperature = 0.2)
  return result[-1]["generated_text"][-1]["content"]

In [47]:
print(answer("What is jason todd watching on TV?"))

Match of the Day


In [48]:
print(answer("What is formula 1"))

I don't know based on the document


In [53]:
print(answer("Who is grace archer's granddad"))

Grandad is Archer's grandfather.


In [55]:
import gradio as gr
def respond(question , gradio_history):
  return answer(question)
demo = gr.ChatInterface(
    respond ,
    title = "The Silent Man - Questionaire",
    description = "A small open-source chatbot running on Qwen2.5-1.5B.",
    examples=["Who is Jason Todd" , "What was jason Todd watching on the TV"]
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://956b4890ef5c3714b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
